# FETCH LATEST REPO VERSION

In [1]:
%cd /content/FLFrameWork
!git pull --ff-only

[Errno 2] No such file or directory: '/content/FLFrameWork'
/content
fatal: not a git repository (or any of the parent directories): .git


# Run FLFrameWork on Colab

Select **Runtime → Change runtime type → GPU**, then run the cells in order. Use a short run first to measure the assigned GPU's speed. This notebook runs one method at a time.

Source code can come from GitHub or the prepared project ZIP. Datasets and active CSV logs stay on the Colab disk; completed CSVs and figures are copied to Google Drive. If Python raises an error or you interrupt execution, the training cell also tries to copy its partial CSV. A disconnected or forcibly terminated runtime may lose work since the last copy. CSVs are not training checkpoints, so they cannot resume an interrupted run.


Check that Colab assigned an NVIDIA GPU and show its name before preparing the project.


In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU under Runtime > Change runtime type, then reconnect.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

GPU: Tesla T4
PyTorch: 2.11.0+cu130


Clone your GitHub repository, or leave its URL empty and upload the prepared FLFrameWork-Colab.zip file.


In [3]:
from pathlib import Path
import io
import os
import subprocess
import sys
import zipfile

REPOSITORY_URL = "https://github.com/s4meer-git/BPDC_FL_Framework.git"
PROJECT_ROOT = Path("/content/FLFrameWork")

if REPOSITORY_URL:
    if (PROJECT_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    from google.colab import files

    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload one FLFrameWork-Colab.zip file.")

    with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
        archive.extractall("/content")
    del uploaded

if not (PROJECT_ROOT / "main.py").is_file():
    raise FileNotFoundError("The project must contain /content/FLFrameWork/main.py.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project:", PROJECT_ROOT)

Project: /content/FLFrameWork


Install the small notebook dependencies and check torchvision while keeping Colab’s existing CUDA-enabled PyTorch installation.


In [4]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "JPNotebooks/requirements.txt"],
    check=True,
)

import torchvision
import pandas as pd

print("torchvision:", torchvision.__version__)

torchvision: 0.26.0+cu130


Connect Google Drive so the method CSVs and analysis PDFs have a place to remain after this runtime ends.


In [5]:
from google.colab import drive
import shutil

_drive_mount = "/content/drive"
drive.mount(_drive_mount)
DRIVE_ROOT = Path(_drive_mount) / "MyDrive" / "FLFrameWork"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


Choose the dataset, method and experiment settings; change the comparison name whenever you start a different experiment.


In [18]:
from configuration import configuration

config = configuration()
config.dataset = "cifar10"
config.method = "fairmedian"
config.num_of_clients = 31
config.rounds = 40
config.local_epochs = 5
config.batch_size = 64
config.fairmedian_batch_size = 64
config.lr = 0.01
config.momentum = 0.5 if config.dataset == "mnist" else 0.9
config.dirichlet_alpha = 0.5
config.seed = 42


# FedCDA starts history search after these warmup rounds; 50 means a 40-round run uses warmup only.
config.fedcda_warmup = 50

config.model = {
    "mnist": "cnn_mnist",
    "cifar10": "cnn_cifar10",
    "tinyimagenet": "resnet18",
}[config.dataset]

COMPARISON_NAME = "experiment_1"
folder_name = f"{COMPARISON_NAME}_{config.dataset}"
RESULTS_FOLDER = PROJECT_ROOT / "Results" / folder_name
SAVED_RESULTS = DRIVE_ROOT / "Results" / folder_name
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS.mkdir(parents=True, exist_ok=True)


# Restore other methods from this comparison so the analysis can include runs from earlier Colab sessions.
shutil.copytree(SAVED_RESULTS, RESULTS_FOLDER, dirs_exist_ok=True)

print("Device:", config.device)
print("Method:", config.method, "Dataset:", config.dataset)
print("Saved CSV folder:", SAVED_RESULTS)

Device: cuda
Method: fairmedian Dataset: cifar10
Saved CSV folder: /content/drive/MyDrive/FLFrameWork/Results/experiment_1_cifar10


Download Tiny ImageNet onto the Colab disk when selected; MNIST and CIFAR-10 are downloaded automatically by main.py.


In [8]:
if config.dataset == "tinyimagenet":
    from torchvision.datasets.utils import download_and_extract_archive

    tiny_folder = PROJECT_ROOT / "data" / "tiny-imagenet-200"
    if not (tiny_folder / "train").is_dir():
        download_and_extract_archive(
            "https://cs231n.stanford.edu/tiny-imagenet-200.zip",
            download_root=str(PROJECT_ROOT / "data"),
        )

    print("Tiny ImageNet:", tiny_folder)
else:
    print(config.dataset, "will download when training starts.")

mnist will download when training starts.


Run the selected method, measure its elapsed time and copy its CSV to Drive when Python finishes or reports an interruption.


In [17]:
import copy
import time
from main import main


# A fresh copy avoids retaining algorithm state if this training cell is run again.
run_config = copy.deepcopy(config)
started = time.perf_counter()

try:
    metrics = main(run_config, output_directory=RESULTS_FOLDER)
finally:
    csv_path = RESULTS_FOLDER / f"{run_config.method.lower()}.csv"
    if csv_path.exists():
        try:
            shutil.copy2(csv_path, SAVED_RESULTS / csv_path.name)
            print("Copied CSV to Drive:", SAVED_RESULTS / csv_path.name)
        except OSError as error:
            print("Drive copy failed; the CSV is still on the Colab disk:", csv_path)
            print(error)

    print("Elapsed minutes:", round((time.perf_counter() - started) / 60, 2))

Device: cuda
Dataset: mnist
Method: fairmedian
Clients: 31
Rounds: 40
Training batch size: 64
FairMedian uses one minibatch gradient per client per round.
Training samples: 60000
Testing samples: 10000
Client 0 training samples: 803 testing samples: 236
Client 1 training samples: 1910 testing samples: 260
Client 2 training samples: 2295 testing samples: 197
Client 3 training samples: 2094 testing samples: 271
Client 4 training samples: 2459 testing samples: 254
Client 5 training samples: 2037 testing samples: 449
Client 6 training samples: 2306 testing samples: 320
Client 7 training samples: 2934 testing samples: 281
Client 8 training samples: 2467 testing samples: 223
Client 9 training samples: 2667 testing samples: 437
Client 10 training samples: 1120 testing samples: 273
Client 11 training samples: 3408 testing samples: 248
Client 12 training samples: 1896 testing samples: 203
Client 13 training samples: 1028 testing samples: 231
Client 14 training samples: 885 testing samples: 481


Enter the comparison name below to find its dataset folders in Drive Results and save the three comparison PDFs beside each folder’s CSVs; rerunning replaces those PDFs.


In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd


# These are the folders used by this notebook's project setup and Google Drive mount.
PROJECT_ROOT = Path("/content/FLFrameWork")
DRIVE_ROOT = Path("/content/drive/MyDrive/FLFrameWork")

if not (PROJECT_ROOT / "analysis" / "results.py").is_file():
    raise FileNotFoundError("Run the project setup cell first to download the framework into /content/FLFrameWork.")
if not (DRIVE_ROOT.parent).is_dir():
    raise FileNotFoundError("Run the Google Drive mount cell first, then run this analysis cell again.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.results import read_run, training_losses

notebook = json.loads((PROJECT_ROOT / "JPNotebooks" / "Analysis.ipynb").read_text())
plot_roles = ("accuracy", "round_loss", "client_loss")


# Enter the comparison name; its dataset folders are found automatically below.
COMPARISON_NAME = "experiment_1"
results_root = DRIVE_ROOT / "Results"

if not results_root.is_dir():
    raise FileNotFoundError(f"Results folder not found: {results_root}")

folders = sorted(
    folder for folder in results_root.iterdir()
    if folder.is_dir() and folder.name.startswith(COMPARISON_NAME + "_")
    and any(folder.glob("*.csv"))
)
if not folders:
    raise FileNotFoundError(f"No CSV folders for {COMPARISON_NAME} found in {results_root}")

for results_folder in folders:


    # Read the recorded dataset, so folder names such as cifar10 and cifar_10 both work.
    try:
        csv_data = {path.name: pd.read_csv(path) for path in sorted(results_folder.glob("*.csv"))}
        datasets = {str(read_run(data, name)["dataset"]).strip().lower() for name, data in csv_data.items()}
        if len(datasets) != 1:
            raise ValueError("This folder contains results from different datasets.")
        dataset = datasets.pop()
        training_losses(csv_data)
    except (ValueError, KeyError) as error:
        print("Skipped folder:", results_folder.name, "-", error)
        continue

    plot_cells = {
        cell["metadata"]["analysis_role"]: cell
        for cell in notebook["cells"]
        if cell["cell_type"] == "code"
        and cell.get("metadata", {}).get("dataset") == dataset
        and cell.get("metadata", {}).get("analysis_role") in plot_roles
    }
    if set(plot_cells) != set(plot_roles):
        print("Skipped folder:", results_folder.name, "- no analysis cells for", dataset)
        continue


    # Each folder gets its own CSVs, plots and PDF destination; no training settings are needed here.
    analysis_data = {
        "PROJECT_ROOT": PROJECT_ROOT,
        "RESULTS_FOLDER": results_folder,
        "csv_data": csv_data,
        "PDF_FOLDER": results_folder,
    }
    print("Analysing:", results_folder.name, "Dataset:", dataset)
    for role in plot_roles:
        exec("".join(plot_cells[role]["source"]), analysis_data)

    print("Saved PDFs:", results_folder)


To run another method, change `config.method` in the settings cell and rerun from that cell onward, keeping the same comparison name. Choose a new comparison name when changing the experiment settings. Each dataset has its own folder, and rerunning a method replaces that method's CSV.

For GitHub updates, stop training, push your local changes, restart the Colab Python session, and run setup again. The source cell pulls the new commit. A restart discards in-memory training state; this framework has no checkpoint/resume support. Avoid editing the same source files independently in Colab: `git pull --ff-only` deliberately stops on conflicting local edits.

FairMedian uses one minibatch per client and needs an odd client count. Its local-epoch setting does not apply. FairRFL currently keeps the authors' hardcoded local momentum of 0.9, including on MNIST; setting `config.momentum` does not override it.
